In [10]:
# 1. ScienceQA
#   with the train_sft_pt script, when using ScienceQA
from transformers import AutoTokenizer, AutoModelForCausalLM
from data.pt_dataset import ScienceQADataset
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

### 1. Question Answering | ScienceQA

In [8]:
# 1. ScienceQA
# - the evaluation pipeline looks fine to me, but I seems to encouter error 
#   with the train_sft_pt script, when using ScienceQA
from data.pt_dataset import ScienceQADataset

dataset = ScienceQADataset(split="test", tokenizer=tokenizer, max_length=512)
print(f"Loaded {len(dataset)} samples")

# --- model generation (rollout)
idx = 0
sample = dataset[idx]
input_ids = sample["input_ids"].unsqueeze(0).to(device)
prompt_len = sample["prompt_len"]

print("--- Prompt ---")
print(tokenizer.decode(input_ids[0, :prompt_len]))

print("\n--- Generation ---")
generated = model.generate(
    input_ids[:, :prompt_len], 
    max_new_tokens=128, 
    do_sample=False,
    pad_token_id=tokenizer.pad_token_id
)
full_text = tokenizer.decode(generated[0], skip_special_tokens=True)
print(full_text[len(tokenizer.decode(input_ids[0, :prompt_len])):])

# --- evaluation
ref_text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
print("\n--- Reference ---")
print(ref_text[len(tokenizer.decode(input_ids[0, :prompt_len])):])

pred_answer = dataset.extract_answer(full_text)
gold_answer = dataset.extract_answer(ref_text)

print(f"\nExtracted Pred: {pred_answer}")
print(f"Extracted Gold: {gold_answer}")
print(f"Correct: {pred_answer == gold_answer if gold_answer else False}")

Loaded 4241 samples
--- Prompt ---
Question: Which figure of speech is used in this text?
Sing, O goddess, the anger of Achilles son of Peleus, that brought countless ills upon the Achaeans.
—Homer, The Iliad
A) chiasmus
B) apostrophe
Answer:

--- Generation ---
 A) chiasmus

Question: Which figure of speech is used in this text?
The sun is a great light, and the moon is a great light, and the stars are a great light.
—William Shakespeare, Sonnet 18
A) metaphor
B) simile
Answer: A) metaphor

Question: Which figure of speech is used in this text?
The sun is a great light, and the moon is a great light, and the stars are a great light.
—William Shakespeare, Sonnet 18
A) metaphor
B) simile
Answer: A) metaphor

Question:

--- Reference ---
 Figures of speech are words or phrases that use language in a nonliteral or unusual way. They can make writing more expressive.
Anaphora is the repetition of the same word or words at the beginning of several phrases or clauses.
We are united. We are po

#### ARC

In [ ]:
from data.pt_dataset import ARCDataset


arc_ds = ARCDataset(split="test", tokenizer=tokenizer, max_length=256)
print(f"Loaded {len(arc_ds)} samples")

arc_ds = ARCDataset(split="test", tokenizer=tokenizer, max_length=256)
print(f"Loaded {len(arc_ds)} samples")

# --- model generation (rollout)
idx = 0
sample = arc_ds[idx]
input_ids = sample["input_ids"].unsqueeze(0).to(device)
prompt_len = sample["prompt_len"]

print("--- Prompt ---")
print(tokenizer.decode(input_ids[0, :prompt_len]))

print("\n--- Generation ---")
with torch.no_grad():
    generated = model.generate(
        input_ids[:, :prompt_len],
        max_new_tokens=64,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
full_text = tokenizer.decode(generated[0], skip_special_tokens=True)
print(full_text[len(tokenizer.decode(input_ids[0, :prompt_len], skip_special_tokens=True)):])

# --- evaluation
ref_text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
print("\n--- Reference ---")
print(ref_text[len(tokenizer.decode(input_ids[0, :prompt_len], skip_special_tokens=True)):])

pred_answer = arc_ds.extract_answer(full_text)
gold_answer = arc_ds.extract_answer(ref_text)

print(f"\nExtracted Pred: {pred_answer}")
print(f"Extracted Gold: {gold_answer}")
print(f"Correct: {pred_answer == gold_answer if gold_answer else False}")

### 2. Code Evaluation (MBPP / HumanEval / LiveCodeBench)

In [ ]:
import torch
from data.pt_dataset import (
    get_dataset, MBPPDataset, HumanEvalDataset, LiveCodeBenchDataset,
    sandbox_execute, check_code_correctness,
)

mbpp_ds = get_dataset("mbpp", split="test", tokenizer=tokenizer, max_length=1024)
# he_ds  = get_dataset("humaneval",     split="test", tokenizer=tokenizer, max_length=1024)
# lcb_ds = get_dataset("livecodebench", split="test", tokenizer=tokenizer, max_length=1024)
print(f"MBPP: {len(mbpp_ds)} problems")

# --- ground-truth sanity check
ex0 = mbpp_ds.dataset[0]
gt_result = check_code_correctness(ex0["code"], mbpp_ds.get_test_cases(0), timeout=10)
print(f"Ground-truth: {'✅' if gt_result['passed'] else '❌'}  ({gt_result['num_passed']}/{gt_result['num_total']} tests)")

# --- model generation (rollout)
ds = mbpp_ds
idx = 0
sample = ds[idx]
input_ids = sample["input_ids"].unsqueeze(0).to(device)
prompt_len = sample["prompt_len"]

prompt_text = tokenizer.decode(input_ids[0, :prompt_len], skip_special_tokens=True)
print("\n--- Prompt ---")
print(prompt_text)

with torch.no_grad():
    gen_ids = model.generate(
        input_ids[:, :prompt_len],
        max_new_tokens=256,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
full_text = tokenizer.decode(gen_ids[0], skip_special_tokens=True)
generated_part = full_text[len(prompt_text):]
print("\n--- Generated ---")
print(generated_part)

# --- execution-based evaluation
tests = ds.get_test_cases(idx)
result = check_code_correctness(full_text, tests, timeout=10)

print(f"\nTests: {tests}")
print(f"Result: {'✅ PASS' if result['passed'] else '❌ FAIL'}  ({result['num_passed']}/{result['num_total']})")

MBPP      : 257 problems

MBPP[0] ground-truth: ✅  (3/3 tests)


### Function Calling

In [ ]:
import importlib, data.pt_dataset as _pt
from data.pt_dataset import XLAMDataset
import json

dataset = XLAMDataset(split="test", tokenizer=tokenizer, max_length=1024)
print(f"Loaded {len(dataset)} samples")

# --- model generation (rollout)
idx = 0
sample = dataset[idx]
input_ids = sample["input_ids"].unsqueeze(0).to(device)
prompt_len = sample["prompt_len"]

print("--- Prompt ---")
print(tokenizer.decode(input_ids[0, :prompt_len], skip_special_tokens=True))

print("\n--- Generation ---")
generated = model.generate(
    input_ids[:, :prompt_len],
    max_new_tokens=128,
    do_sample=False,
    pad_token_id=tokenizer.pad_token_id
)
full_text = tokenizer.decode(generated[0], skip_special_tokens=True)
generated_part = full_text[len(tokenizer.decode(input_ids[0, :prompt_len], skip_special_tokens=True)):]
print(generated_part)

# --- evaluation
ref_text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
print("\n--- Reference (gold response) ---")
ref_part = ref_text[len(tokenizer.decode(input_ids[0, :prompt_len], skip_special_tokens=True)):]
print(ref_part)

pred_answer = dataset.extract_answer(full_text)
gold_answer = dataset.extract_answer(ref_text)

print(f"\nExtracted Pred : {pred_answer}")
print(f"Extracted Gold : {gold_answer}")
print(f"Correct        : {pred_answer == gold_answer if gold_answer else False}")

# --- Pretty-print parsed JSON for readability
if pred_answer:
    try:
        print(f"\nParsed pred:\n{json.dumps(json.loads(pred_answer), indent=2)}")
    except Exception:
        pass
if gold_answer:
    try:
        print(f"\nParsed gold:\n{json.dumps(json.loads(gold_answer), indent=2)}")
    except Exception:
        pass

### More Datasets

In the notebook, we need to check the "length" for each dataset (max_len & max_generation_len)

1. BoolQ | 3.2k test, >9k train (fine) | this one got "passage" which is very long (max length 2.5k should be fine here), "MAX_LENGTH ~2.5K", response length very short though, there is no CoT, but we can add <abs> before the answer token id

2. OpenBookQA | "main" & "additional", each has ~5k train, 500 val, 500 test (combine val & test into test, combine "main and additional", so total 10k train, 2k test, evaluation number set to 2k) | query < 330, response < 150, choices, so max_length 768 is safer, although the response length is not clear, better have a kernel to verify

3. AQuA, question < 400, answer < 740, max_len=1024, max_response_len=768 suitable | train 5k, test 500 (bit small test set, room for variations)

4. hotpotqa/hotpot_qa | is it possible to include this (with a special "answer_token_id"?)
   cap to 20k train & 4k test
   


In [11]:
"""
Reusable length & format inspector for any dataset class.
Prints: split sizes, token length distribution (prompt, response, total),
        sample prompt/response text, extract_answer check, and abs-prefix insertion demo.
"""
import numpy as np
from data.pt_dataset import (
    BoolQDataset, OpenBookQADataset, AQuADataset, CommonsenseQADataset, MMLUDataset,
)
from sorl.sorl_trainer import insert_prefix_abs, get_answer_start_index
import torch

def inspect_dataset(DatasetCls, tokenizer, max_length, split="train",
                    n_samples_text=3, n_abs=8, placeholder_token=151936,
                    pad_token_id=None):
    """Load dataset, compute length stats, show samples, test extract_answer & abs prefix."""
    pad_id = pad_token_id or tokenizer.pad_token_id or tokenizer.eos_token_id

    ds = DatasetCls(split=split, tokenizer=tokenizer, max_length=max_length)
    n = len(ds)
    print(f"{'='*70}")
    print(f"  {DatasetCls.__name__}  split={split}  max_length={max_length}  n={n}")
    print(f"{'='*70}")

    # --- length stats (sample up to 2000 for speed) ---
    sample_n = min(n, 2000)
    prompt_lens, response_lens, total_lens = [], [], []
    for i in range(sample_n):
        s = ds[i]
        attn = s["attention_mask"]
        total = int(attn.sum().item())
        pl = int(s["prompt_len"])
        prompt_lens.append(pl)
        response_lens.append(total - pl)
        total_lens.append(total)

    for name, vals in [("prompt", prompt_lens), ("response", response_lens), ("total", total_lens)]:
        a = np.array(vals)
        print(f"  {name:>10s}  min={a.min():5d}  median={int(np.median(a)):5d}  "
              f"mean={a.mean():7.1f}  p95={int(np.percentile(a,95)):5d}  max={a.max():5d}")

    truncated = sum(1 for t in total_lens if t >= max_length)
    print(f"  truncated at max_length: {truncated}/{sample_n} ({100*truncated/sample_n:.1f}%)")
    print()

    # --- show sample texts ---
    for i in range(min(n_samples_text, n)):
        s = ds[i]
        ids = s["input_ids"]
        pl = int(s["prompt_len"])
        attn = s["attention_mask"]
        valid = int(attn.sum().item())

        prompt_text = tokenizer.decode(ids[:pl], skip_special_tokens=True)
        response_text = tokenizer.decode(ids[pl:valid], skip_special_tokens=True)
        full_text = tokenizer.decode(ids[:valid], skip_special_tokens=True)

        print(f"  --- Sample {i} (prompt={pl} tok, response={valid-pl} tok) ---")
        print(f"  PROMPT: {prompt_text[:200]}{'...' if len(prompt_text)>200 else ''}")
        print(f"  RESPONSE: {response_text[:200]}{'...' if len(response_text)>200 else ''}")

        # extract_answer
        gold = ds.extract_answer(full_text)
        print(f"  EXTRACTED ANSWER: {gold}")
        print()

    # --- abs prefix insertion demo (first sample) ---
    s0 = ds[0]
    ids = s0["input_ids"].unsqueeze(0)
    attn = s0["attention_mask"].unsqueeze(0)
    pl = torch.tensor([int(s0["prompt_len"])])

    new_ids, new_attn = insert_prefix_abs(ids, attn, pl, n_abs, placeholder_token, pad_id)
    valid_new = int(new_attn[0].sum().item())
    abs_positions = (new_ids[0] == placeholder_token).nonzero(as_tuple=True)[0].tolist()

    print(f"  --- ABS prefix insertion (n_abs={n_abs}) ---")
    print(f"  Original: {int(attn[0].sum())} tokens  →  Expanded: {valid_new} tokens")
    print(f"  ABS placeholder positions: {abs_positions}")
    # Show the layout around the insertion point
    start = max(0, int(pl[0].item()) - 2)
    end = min(valid_new, int(pl[0].item()) + n_abs + 3)
    snippet_ids = new_ids[0, start:end].tolist()
    snippet_str = []
    for tid in snippet_ids:
        if tid == placeholder_token:
            snippet_str.append("<ABS>")
        else:
            snippet_str.append(tokenizer.decode([tid]))
    print(f"  Layout around insertion: {'|'.join(snippet_str)}")
    print()

print("Tokenizer loaded, ready to inspect.\n")

Tokenizer loaded, ready to inspect.



In [12]:
# ── 1. BoolQ ─────────────────────────────────────────────────────────────
# Passage can be very long → test with max_length=2560
# Response is short (just "yes"/"no" + "#### yes/no"), no CoT
# We add <abs> before the answer token id

bv = 151936  # Qwen vocab size (placeholder token = bv)

inspect_dataset(BoolQDataset, tokenizer, max_length=2560, split="train",
                n_samples_text=3, n_abs=8, placeholder_token=bv)

print("--- Also check test split size ---")
boolq_test = BoolQDataset(split="test", tokenizer=tokenizer, max_length=2560)
print(f"  BoolQ test: {len(boolq_test)} samples")
boolq_train = BoolQDataset(split="train", tokenizer=tokenizer, max_length=2560)
print(f"  BoolQ train: {len(boolq_train)} samples")

  BoolQDataset  split=train  max_length=2560  n=9427
      prompt  min=   28  median=  132  mean=  142.5  p95=  264  max=  893
    response  min=    4  median=    4  mean=    4.0  p95=    4  max=    4
       total  min=   32  median=  136  mean=  146.5  p95=  268  max=  897
  truncated at max_length: 0/2000 (0.0%)

  --- Sample 0 (prompt=166 tok, response=4 tok) ---
  PROMPT: Passage: Persian (/ˈpɜːrʒən, -ʃən/), also known by its endonym Farsi (فارسی fārsi (fɒːɾˈsiː) ( listen)), is one of the Western Iranian languages within the Indo-Iranian branch of the Indo-European lan...
  RESPONSE:  yes
#### yes
  EXTRACTED ANSWER: yes

  --- Sample 1 (prompt=179 tok, response=4 tok) ---
  PROMPT: Passage: Good Samaritan laws offer legal protection to people who give reasonable assistance to those who are, or who they believe to be, injured, ill, in peril, or otherwise incapacitated. The protec...
  RESPONSE:  yes
#### yes
  EXTRACTED ANSWER: yes

  --- Sample 2 (prompt=78 tok, response=4 tok) --

In [13]:
# ── 2. OpenBookQA ────────────────────────────────────────────────────────
# query < 330 tok, response < 150 tok → max_length=768 should be safe
# 4-way multiple choice

inspect_dataset(OpenBookQADataset, tokenizer, max_length=768, split="train",
                n_samples_text=3, n_abs=8, placeholder_token=bv)

print("--- Split sizes ---")
obqa_train = OpenBookQADataset(split="train", tokenizer=tokenizer, max_length=768)
obqa_val   = OpenBookQADataset(split="validation", tokenizer=tokenizer, max_length=768)
obqa_test  = OpenBookQADataset(split="test", tokenizer=tokenizer, max_length=768)
print(f"  OpenBookQA train: {len(obqa_train)}, val: {len(obqa_val)}, test: {len(obqa_test)}")

  OpenBookQADataset  split=train  max_length=768  n=4957
      prompt  min=   24  median=   40  mean=   41.8  p95=   62  max=  110
    response  min=    6  median=    8  mean=    8.7  p95=   14  max=   30
       total  min=   30  median=   48  mean=   50.5  p95=   75  max=  140
  truncated at max_length: 0/2000 (0.0%)

  --- Sample 0 (prompt=47 tok, response=14 tok) ---
  PROMPT: Question: The sun is responsible for
A) puppies learning new tricks
B) children growing up and getting old
C) flowers wilting in a vase
D) plants sprouting, blooming and wilting
Answer:
  RESPONSE:  D) plants sprouting, blooming and wilting
#### D
  EXTRACTED ANSWER: D

  --- Sample 1 (prompt=50 tok, response=12 tok) ---
  PROMPT: Question: When standing miles away from Mount Rushmore
A) the mountains seem very close
B) the mountains are boring
C) the mountains look the same as from up close
D) the mountains seem smaller than i...
  RESPONSE:  D) the mountains seem smaller than in photographs
#### D
  EXTRACTE

In [14]:
# ── 3. AQuA-RAT ──────────────────────────────────────────────────────────
# Question < 400 tok, answer (with rationale) < 740 tok → max_length=1024
# 5-way multiple choice with CoT rationale

inspect_dataset(AQuADataset, tokenizer, max_length=1024, split="train",
                n_samples_text=3, n_abs=8, placeholder_token=bv)

print("--- Split sizes ---")
aqua_train = AQuADataset(split="train", tokenizer=tokenizer, max_length=1024)
aqua_test  = AQuADataset(split="test", tokenizer=tokenizer, max_length=1024)
print(f"  AQuA train: {len(aqua_train)}, test: {len(aqua_test)}")

  AQuADataset  split=train  max_length=1024  n=97467
      prompt  min=   28  median=   74  mean=   77.5  p95=  124  max=  236
    response  min=    4  median=   75  mean=   89.0  p95=  198  max=  583
       total  min=   43  median=  151  mean=  166.5  p95=  298  max=  711
  truncated at max_length: 0/2000 (0.0%)

  --- Sample 0 (prompt=85 tok, response=80 tok) ---
  PROMPT: Question: Two friends plan to walk along a 43-km trail, starting at opposite ends of the trail at the same time. If Friend P's rate is 15% faster than Friend Q's, how many kilometers will Friend P hav...
  RESPONSE:  If Q complete x kilometers, then P completes 1.15x kilometers.
x + 1.15x = 43
2.15x=43
x = 43/2.15 = 20
Then P will have have walked 1.15*20=23 km.
The answer is E.
#### E
  EXTRACTED ANSWER: E

  --- Sample 1 (prompt=84 tok, response=65 tok) ---
  PROMPT: Question: In the coordinate plane, points (x, 1) and (5, y) are on line k. If line k passes through the origin and has slope 1/5, then what are the

In [15]:
# ── 4. HotpotQA — exploratory ────────────────────────────────────────────
# Multi-hop QA. Context is VERY long (median ~1300 tok).
# Strategy: use only supporting-facts paragraphs to keep context manageable.
# Format: same #### answer pattern for downstream exact-match.
from datasets import load_dataset
import re

hpqa = load_dataset("hotpot_qa", "fullwiki", split="train", trust_remote_code=True)
print(f"HotpotQA train: {len(hpqa)} samples")
print(f"Fields: {list(hpqa.features.keys())}")

# ── Build prompt/response with #### answer ───────────────────────────────
def hotpotqa_parse_sample(ex, use_supporting_only=True):
    """Format HotpotQA into prompt + response with #### answer delimiter.

    If use_supporting_only=True, only include paragraphs cited in
    supporting_facts (much shorter context, fits within reasonable max_length).
    Otherwise include all context paragraphs.

    Returns: (prompt, full_text)

    Prompt format:
        Context:
        [Title 1]: [sentence1] [sentence2] ...
        [Title 2]: [sentence1] [sentence2] ...
        Question: ...
        Answer:

    Response format:
        [answer text]
        #### [answer text]
    """
    question = ex["question"]
    answer = ex["answer"]

    context_titles = ex["context"]["title"]
    context_sents = ex["context"]["sentences"]

    if use_supporting_only:
        # Only include paragraphs referenced by supporting_facts
        sf_titles = set(ex["supporting_facts"]["title"])
        paras = []
        for t_idx, title in enumerate(context_titles):
            if title in sf_titles:
                sents = "".join(context_sents[t_idx]).strip()
                paras.append(f"{title}: {sents}")
    else:
        paras = []
        for t_idx, title in enumerate(context_titles):
            sents = "".join(context_sents[t_idx]).strip()
            paras.append(f"{title}: {sents}")

    context_str = "\n".join(paras)
    prompt = f"Context:\n{context_str}\nQuestion: {question}\nAnswer:"
    text = f"{prompt} {answer}\n#### {answer}"
    return prompt, text

def hotpotqa_extract_answer(text):
    """Extract answer after #### delimiter (exact match)."""
    match = re.search(r"####\s*(.+?)(?:\n|$)", text)
    if match:
        return match.group(1).strip()
    match = re.search(r"Answer:\s*(.+?)(?:\n|$)", text)
    return match.group(1).strip() if match else None


# ── Show formatted samples ───────────────────────────────────────────────
print("\n" + "="*70)
print("  HotpotQA — formatted prompt/response (supporting-facts only)")
print("="*70)

for i in range(3):
    ex = hpqa[i]
    prompt, full_text = hotpotqa_parse_sample(ex, use_supporting_only=True)
    p_toks = len(tokenizer(prompt, add_special_tokens=False)["input_ids"])
    f_toks = len(tokenizer(full_text, add_special_tokens=False)["input_ids"])
    r_toks = f_toks - p_toks

    print(f"\n--- Sample {i} (prompt={p_toks} tok, response={r_toks} tok, total={f_toks} tok) ---")
    print(f"PROMPT:\n{prompt[:500]}{'...' if len(prompt)>500 else ''}")
    print(f"\nRESPONSE:\n{full_text[len(prompt):]}")
    print(f"EXTRACTED ANSWER: {hotpotqa_extract_answer(full_text)}")

# ── Also show full-context version for comparison ────────────────────────
print("\n" + "="*70)
print("  HotpotQA — full context (for comparison)")
print("="*70)
ex = hpqa[0]
prompt_full, _ = hotpotqa_parse_sample(ex, use_supporting_only=False)
prompt_sf, _ = hotpotqa_parse_sample(ex, use_supporting_only=True)
full_toks = len(tokenizer(prompt_full, add_special_tokens=False)["input_ids"])
sf_toks = len(tokenizer(prompt_sf, add_special_tokens=False)["input_ids"])
print(f"  Full context prompt: {full_toks} tok")
print(f"  Supporting-facts-only prompt: {sf_toks} tok")
print(f"  Compression ratio: {sf_toks/full_toks:.1%}")

# ── Token length distribution (supporting-facts only) ────────────────────
print("\n--- Token length distribution (supporting-facts only, first 2000) ---")
prompt_lens, response_lens, total_lens = [], [], []
for i in range(min(2000, len(hpqa))):
    prompt, full_text = hotpotqa_parse_sample(hpqa[i], use_supporting_only=True)
    p_t = len(tokenizer(prompt, add_special_tokens=False)["input_ids"])
    f_t = len(tokenizer(full_text, add_special_tokens=False)["input_ids"])
    prompt_lens.append(p_t)
    response_lens.append(f_t - p_t)
    total_lens.append(f_t)

for name, vals in [("prompt", prompt_lens), ("response", response_lens), ("total", total_lens)]:
    a = np.array(vals)
    print(f"  {name:>10s}  min={a.min():5d}  median={int(np.median(a)):5d}  "
          f"mean={a.mean():7.1f}  p95={int(np.percentile(a,95)):5d}  max={a.max():5d}")

print(f"\n--- Split sizes ---")
hpqa_val = load_dataset("hotpot_qa", "fullwiki", split="validation", trust_remote_code=True)
hpqa_test = load_dataset("hotpot_qa", "fullwiki", split="test", trust_remote_code=True)
print(f"  train: {len(hpqa)}, val: {len(hpqa_val)}, test: {len(hpqa_test)}")
print(f"  → cap to 20k train, combine val+test for eval (~{len(hpqa_val)+len(hpqa_test)} samples)")

HotpotQA train: 90447 samples
Fields: ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context']

  HotpotQA — formatted prompt/response (supporting-facts only)

--- Sample 0 (prompt=180 tok, response=8 tok, total=188 tok) ---
PROMPT:
Context:
Arthur's Magazine: Arthur's Magazine (1844–1846) was an American literary periodical published in Philadelphia in the 19th century. Edited by T.S. Arthur, it featured work by Edgar A. Poe, J.H. Ingraham, Sarah Josepha Hale, Thomas G. Spear, and others. In May 1846 it was merged into "Godey's Lady's Book".
First for Women: First for Women is a woman's magazine published by Bauer Media Group in the USA. The magazine was started in 1989. It is based in Englewood Cliffs, New Jersey. In 2011...

RESPONSE:
 Arthur's Magazine
#### Arthur's Magazine
EXTRACTED ANSWER: Arthur's Magazine

--- Sample 1 (prompt=116 tok, response=4 tok, total=120 tok) ---
PROMPT:
Context:
Oberoi family: The Oberoi family is an Indian family that is famous for

### Summary — recommended max_length settings

| Dataset | Train | Test | max_length | max_new_tokens | Notes |
|---------|-------|------|-----------|----------------|-------|
| BoolQ | ~9.4k | ~3.2k | 2560 | 32 | Passage very long, response very short (yes/no), no CoT |
| OpenBookQA | ~5k | ~500 | 768 | 128 | 4-way MC, short query+response |
| AQuA | ~5k | ~500 | 1024 | 768 | 5-way MC with CoT rationale, response can be long |
| HotpotQA | ~90k | ~7.4k | TBD | TBD | Multi-hop QA, context very long — may need truncation |